In [2]:
import os, json, yaml, time
import datetime
import urllib3
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Proj
from scipy.spatial import cKDTree
from itertools import product
from scipy.stats import bootstrap, ecdf
from shapely.geometry import Point, MultiPolygon
import multiprocessing
from functools import partial
from multiprocessing import Pool


### List and verify metrics files

In [3]:
home_loc_df = pd.read_parquet('/glade/derecho/scratch/cbecker/CAT_home_loc_20251010/home_loc_by_row_group.parquet')

In [47]:
home_loc_df.shape

(42039428, 5)

In [4]:
import geopandas as gpd
from shapely.geometry import Point

# polygons: counties
counties = gpd.read_file('/glade/derecho/scratch/cbecker/California_County_Boundaries_6550485670014028237.geojson')
counties = counties.to_crs("EPSG:4326")

# points: millions of lon/lat pairs
import pandas as pd
import numpy as np

lats = home_loc_df['lat'].values
lons = home_loc_df['lng'].values
points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(lons, lats), crs="EPSG:4326")

joined = gpd.sjoin(points, counties, how="left", predicate="within")
joined

,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID
0,POINT (-117.20835 33.09914),36.0,37.0,San Diego,SDG,37.0,37,073,None,414826ec-689e-4084-bd03-5195df2748bf
1,POINT (-117.98774 33.71172),29.0,30.0,Orange,ORA,30.0,30,059,None,e9a24c4d-a4ec-40b5-b689-89de5354a669
2,POINT (-121.83142 37.36533),42.0,43.0,Santa Clara,SCL,43.0,43,085,None,2220f7b9-9361-4fb8-9456-872109cf6cf7
3,POINT (-119.79348 36.81353),9.0,10.0,Fresno,FRE,10.0,10,019,None,f57a290a-f11e-4b3b-a53b-e8e4c2284bbb
4,POINT (-118.45496 33.99308),18.0,19.0,Los Angeles,LOS,19.0,19,037,None,3b1e1d69-2b1a-464d-ba43-611c4201b219
...,...,...,...,...,...,...,...,...,...,...
42039423,POINT (-117.90786 33.99553),18.0,19.0,Los Angeles,LOS,19.0,19,037,None,3b1e1d69-2b1a-464d-ba43-611c4201b219
42039424,POINT (-122.64016 38.4561),48.0,49.0,Sonoma,SON,49.0,49,097,None,897e1214-bf0a-456f-bfb4-d90e49c14c13
42039425,POINT (-122.78038 39.06872),16.0,17.0,Lake,LAK,17.0,17,033,None,3490fe5a-a9e4-4997-b746-25abee1b9e23
42039426,POINT (-118.41259 33.77047),18.0,19.0,Los Angeles,LOS,19.0,19,037,None,3b1e1d69-2b1a-464d-ba43-611c4201b219


In [5]:
home_loc_df["County_Name"] = joined["COUNTY_NAME"].values

In [46]:
joined.shape

(42039428, 10)

In [6]:
home_loc_df.to_parquet("/glade/campaign/cisl/aiml/cat_flood/home_loc_and_meta_all.parquet")

In [2]:
base_data_path = "/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121/"
start_date = "2023-03-15"
end_date = "2024-04-29"

date_range = pd.date_range(start=start_date, end=end_date, freq='d')
data_files = date_range.strftime(f"{base_data_path}veraset_%Y-%m-%d_PST_chunked.parquet")
files = []
for i, f in enumerate(data_files):
    if os.path.isfile(f):
        files.append(f)
print(len(files))

92


### Create boolean columns for FFWs and rain amounts during the AR period

#### FFWs

In [7]:
def get_bool(row, polys):
    point = Point(row[["lng_home", "lat_home"]])
    return any([p.contains(point) for p in polys])

In [1]:
for file in files[58:64]:
    df_temp = pd.read_parquet(file)
    s_date = file.split("_PST")[0].split("veraset_")[1]
    e_date = (datetime.datetime.strptime(s_date, '%Y-%m-%d') + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
    url = f"https://mesonet.agron.iastate.edu/api/1/cow.json?wfo=LOX&begints={s_date}T00:00:00Z&enable_shared_border=1&endts={e_date}T00:00:00Z&hailsize=1&wind=58&phenomena=FF&lsrtype=FF&lsrbuffer=30&warningbuffer=1"
    req = urllib3.request("GET", url)
    ffw_warnings = json.loads(req.data)["events"]["features"]
    polygons = [MultiPolygon(w["geometry"]["coordinates"]) for w in ffw_warnings]
    df_temp[f"FFW"] = df_temp.apply(get_bool, polys=polygons, axis=1)
    df_temp.to_parquet(file.replace("analysis_ready_data_20251121", "analysis_ready_data_20251121_metavars"))

NameError: name 'files' is not defined

In [21]:
files_updated = [f.replace("analysis_ready_data_20251121", "analysis_ready_data_20251121_mrms") for f in files]

#### MRMS Variables

In [19]:
def create_tree(mrms_ds):
    # define projection
    proj = Proj(proj="lcc", R=6371229, lat_0=25, lon_0=265, lat_1=25, lat_2=25)
    
    # grid and project MRMS lat-lon coords then create KDTree
    mrms_lon_grid = np.tile(mrms_ds['lon'].values-360, (mrms_ds['lat'].values.shape[0], 1)).T
    mrms_lat_grid = np.tile(mrms_ds['lat'].values, (mrms_ds['lon'].values.shape[0], 1))
    proj_mrms_lon_grid, proj_mrms_lat_grid = np.array(proj(mrms_lon_grid, mrms_lat_grid))
    tree = cKDTree(np.c_[proj_mrms_lon_grid.ravel(), proj_mrms_lat_grid.ravel()])
    return tree, proj, mrms_lon_grid.shape

def get_mrms_var(df_temp, date, idx, mrms_ds, variable, variable_name):

    # grab MRMS data using calculated index
    lon_idx_da = xr.DataArray(idx[:, 0], dims="z")
    lat_idx_da = xr.DataArray(idx[:, 1], dims="z")
    mrms_idx = mrms_ds[variable].sel(time=date).isel(lon=lon_idx_da, lat=lat_idx_da).values
    series = pd.Series(mrms_idx.tolist(), name=variable_name, index=df_temp.index)
    return series

def cycle_through_mrms_variables(file, variables, variable_aggregates):

    mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variables[0]}_2024-03-15T00:00_2024-04-30T00:00_daily_{variable_aggregates[0]}.nc"
    mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
    tree, proj, mrms_lon_grid_shape = create_tree(mrms_ds)
    
    df_temp = pd.read_parquet(file)
    date = df_temp["date"].iloc[0].to_numpy()
    year = df_temp["date"].iloc[0].year
    _, points = tree.query(np.column_stack(proj(df_temp['lng_home'], df_temp['lat_home'])))
    idx = np.column_stack(np.unravel_index(points, mrms_lon_grid_shape))

    for variable, variable_aggregate in product(variables, variable_aggregates):
        mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variable}_{year}-03-15T00:00_{year}-04-30T00:00_daily_{variable_aggregate}.nc"
        mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
        variable_name = f"streamflow_{variable_aggregate}" if "STREAMFLOW" in variable else f"precip_{variable_aggregate}"
        df_temp[f"{variable_name}"] = get_mrms_var(df_temp, date, idx, mrms_ds, variable, variable_name)
    file = file.replace("analysis_ready_data_20251121_metavars", "analysis_ready_data_20251121_mrms")
    df_temp.to_parquet(file)
    
    return None

In [16]:
out_dir = "/glade/campaign/cisl/aiml/cat_flood/analysis_read_data_20251121_mrms"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [20]:
variables = ["MultiSensor_QPE_01H_Pass2_00.00", "FLASH_HP_MAXUNITSTREAMFLOW_00.00"]
variable_aggregates = ["max", "sum"]

n_processors = multiprocessing.cpu_count()
f = partial(cycle_through_mrms_variables,
            variables=variables,
            variable_aggregates=variable_aggregates)

s = time.time()
print('Beginning data processing...')
with Pool(processes=n_processors) as pool:
    result = pool.map_async(f, files_updated[58:64])
    results = result.get()
print(f"{np.round((time.time() - s) / 60, 2)} minutes to complete parallel computation.")
print(len(results))

Beginning data processing...
0.22 minutes to complete parallel computation.
6


In [23]:
ls /glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms

ls: cannot access '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms': No such file or directory


In [36]:
files_updated[58:64]

['/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2024-03-27_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2024-03-28_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2024-03-29_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2024-03-30_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2024-03-31_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2024-04-01_PST_chunked.parquet']

In [37]:
df_temp = pd.read_parquet(files_updated[61])
df_temp

,uid,duration,lat_home,lng_home,zipcode,row_group,county,number_of_visits,distance_straight_line,jump_lengths,radius_of_gyration,num_stops,max_distance_from_home,date,FFW,precip_max,precip_sum,streamflow_max,streamflow_sum
67,0000d3f70d721e99161e7f39154bff9d4319a53f6baad7...,4 days 23:15:49,34.107017,-118.200225,90042,0,LA,15,4.686471,"[0.694696883239104, 0.608335421697375, 1.22487...",1.009130,1,3.468525,2024-03-30,True,5.3,33.6,1.1,8.7
101,0000cca2b41fbb5282e55f919a6ca78d566e5e6d6b8fd3...,1 days 22:46:33,34.058109,-118.047034,91733,0,LA,68,3.292597,"[0.0021208749122438864, 0.023684496086685096, ...",1.025924,2,2.775342,2024-03-30,False,5.4,37.6,1.2,11.8
206,0000c10b4b076e94d56542852a5096f8e0a7d237de8fa3...,2 days 23:30:42,34.238634,-118.474438,91343,0,LA,13,5.044065,"[0.0014528597446924487, 0.007865244920086266, ...",1.123050,2,4.502037,2024-03-30,False,7.4,39.6,1.3,8.7
305,0000971c46d7ccb0d14da20913f0d4738acb557f1c726b...,0 days 08:22:13,33.945946,-118.155960,90241,0,LA,23,0.092681,"[0.003817574843736571, 0.005510592851732759, 0...",0.003546,1,0.006807,2024-03-30,True,6.8,39.7,0.6,6.2
336,000093ca81930e1c247112fceaa58e2d2e3df3e1d438f6...,21 days 11:29:35,34.099681,-117.846496,91724,0,LA,38,72.210663,"[0.004299526646645943, 15.974572617081542, 0.0...",12.564608,3,28.640231,2024-03-30,False,4.6,36.9,1.4,11.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,ffffdc3e82f40400ad08a8978fbf1be18cf12689466633...,2 days 04:24:43,34.186478,-118.425171,91401,75017,LA,2,0.001466,[0.0014662291372841661],0.000763,1,0.001100,2024-03-30,False,6.2,34.8,0.9,5.6
375,ffffa5a7e8c0b8e6d98a832b851277bc3279115353ba87...,4 days 11:43:54,33.892442,-118.358534,90260,75017,LA,14,4.127734,"[0.008331843044791249, 0.025972053853424316, 0...",1.994276,2,4.035528,2024-03-30,True,8.1,37.0,0.5,5.1
500,ffff8c6eca66e179685ada7106011ec94a8e64faf7aa0f...,4 days 11:32:52,34.033876,-118.166443,90022,75017,LA,17,18.414063,"[0.005759152457543746, 0.006981752683202895, 0...",6.917045,2,16.008543,2024-03-30,True,5.4,36.3,0.8,7.8
572,ffff88da7942d1c71e8cc139e95c8ae3d0b36061725b3b...,5 days 14:43:57,33.907301,-118.164314,90723,75017,LA,14,7.802838,"[0.0, 0.005514274774521933, 3.8721042085645787...",1.749946,3,4.825119,2024-03-30,True,5.8,45.0,0.4,4.0


In [38]:
df_temp["precip_sum"].max()

np.float64(91.2)

In [39]:
df_temp["county"].unique()

array(['LA', 'VE', 'SB'], dtype=object)

In [25]:
AR_dates = pd.date_range("2024-03-29", "2024-04-01", freq="1d")

bool_series = []
for date in AR_dates:
    df_temp = pd.read_parquet(files_updated[58].replace("2023-03-27", date.strftime('%Y-%m-%d')))
    bool_series.append(df_temp["FFW"].rename(f"FFW_{date.strftime('%Y-%m-%d')}"))
    for mrms_metric in ['precip_max', 'precip_sum', 'streamflow_max', 'streamflow_sum']:
        bool_series.append(df_temp.apply(lambda x: True if x[mrms_metric] > 0 else False, axis=1).rename(f"{mrms_metric}_{date.strftime('%Y-%m-%d')}"))
bool_df = pd.concat(bool_series, axis=1)

['/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2023-03-15_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2023-03-16_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2023-03-17_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2023-03-18_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2023-03-19_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2023-03-20_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2023-03-21_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2023-03-22_PST_chunked.parquet',
 '/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121_mrms/veraset_2023-03-23_PST_c

In [ ]:
for file in files_updated[58:64]:
    df_temp = pd.read_parquet(file)
    print(df_temp.shape)
    df_temp = pd.merge(df_temp, bool_df, left_index=True, right_index=True, how='left')
    print(df_temp.shape)
df_temp

(790763, 19)


In [26]:
df_temp.shape

(915525, 19)

In [41]:
df_temp

,uid,duration,lat_home,lng_home,zipcode,row_group,county,number_of_visits,distance_straight_line,jump_lengths,radius_of_gyration,num_stops,max_distance_from_home,date,FFW,precip_max,precip_sum,streamflow_max,streamflow_sum
67,0000d3f70d721e99161e7f39154bff9d4319a53f6baad7...,4 days 23:15:49,34.107017,-118.200225,90042,0,LA,15,4.686471,"[0.694696883239104, 0.608335421697375, 1.22487...",1.009130,1,3.468525,2024-03-30,True,5.3,33.6,1.1,8.7
101,0000cca2b41fbb5282e55f919a6ca78d566e5e6d6b8fd3...,1 days 22:46:33,34.058109,-118.047034,91733,0,LA,68,3.292597,"[0.0021208749122438864, 0.023684496086685096, ...",1.025924,2,2.775342,2024-03-30,False,5.4,37.6,1.2,11.8
206,0000c10b4b076e94d56542852a5096f8e0a7d237de8fa3...,2 days 23:30:42,34.238634,-118.474438,91343,0,LA,13,5.044065,"[0.0014528597446924487, 0.007865244920086266, ...",1.123050,2,4.502037,2024-03-30,False,7.4,39.6,1.3,8.7
305,0000971c46d7ccb0d14da20913f0d4738acb557f1c726b...,0 days 08:22:13,33.945946,-118.155960,90241,0,LA,23,0.092681,"[0.003817574843736571, 0.005510592851732759, 0...",0.003546,1,0.006807,2024-03-30,True,6.8,39.7,0.6,6.2
336,000093ca81930e1c247112fceaa58e2d2e3df3e1d438f6...,21 days 11:29:35,34.099681,-117.846496,91724,0,LA,38,72.210663,"[0.004299526646645943, 15.974572617081542, 0.0...",12.564608,3,28.640231,2024-03-30,False,4.6,36.9,1.4,11.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,ffffdc3e82f40400ad08a8978fbf1be18cf12689466633...,2 days 04:24:43,34.186478,-118.425171,91401,75017,LA,2,0.001466,[0.0014662291372841661],0.000763,1,0.001100,2024-03-30,False,6.2,34.8,0.9,5.6
375,ffffa5a7e8c0b8e6d98a832b851277bc3279115353ba87...,4 days 11:43:54,33.892442,-118.358534,90260,75017,LA,14,4.127734,"[0.008331843044791249, 0.025972053853424316, 0...",1.994276,2,4.035528,2024-03-30,True,8.1,37.0,0.5,5.1
500,ffff8c6eca66e179685ada7106011ec94a8e64faf7aa0f...,4 days 11:32:52,34.033876,-118.166443,90022,75017,LA,17,18.414063,"[0.005759152457543746, 0.006981752683202895, 0...",6.917045,2,16.008543,2024-03-30,True,5.4,36.3,0.8,7.8
572,ffff88da7942d1c71e8cc139e95c8ae3d0b36061725b3b...,5 days 14:43:57,33.907301,-118.164314,90723,75017,LA,14,7.802838,"[0.0, 0.005514274774521933, 3.8721042085645787...",1.749946,3,4.825119,2024-03-30,True,5.8,45.0,0.4,4.0


### Calculate Bootsrap average, sign, and p-value

In [ ]:
for mrms_label in ["precip", "streamflow"]:
    mrms_metrics = [c for c in list(df.columns) if any(k in c for k in [f'{mrms_label}_max_', f'{mrms_label}_sum_'])]
    mrms_metrics_bool = [c+b for c in mrms_metrics for b in ["_yes", "_no"]]
    mrms_metrics_bool_bs = [c+bs for c in mrms_metrics_bool for bs in ["_pvalue", "_boot_mean", "_boot_sign"]]
    deltas = [c for c in list(df.columns) if any(k in c for k in ['_z', '_modz'])]
    counties = ["LA", "SB", "VE"]
    bools = ["yes", "no"]
    
    for county in counties:
        print(county)
        df_bs = pd.DataFrame(columns=mrms_metrics_bool_bs, index=deltas)
        for mrms_metric in mrms_metrics:
            print(mrms_metric)
            for delta in deltas:
                for bool in bools:
                    data = df.loc[(df[mrms_metric]==bool) & (df['county']==county)][delta].dropna()
                    if data.shape[0]<2:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = np.nan
                        continue
                    else:
                        res = bootstrap((data,), np.mean, confidence_level=0.95, paired=False, alternative='two-sided')
                        res_ecdf = ecdf(res.bootstrap_distribution)
                        res_ecdf_0 = float(res_ecdf.cdf.evaluate(0))
                        if res_ecdf_0 > 0.5:
                            df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = 1.0 - res_ecdf_0
                        else:
                            df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = res_ecdf_0
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_mean"] = np.mean(res.bootstrap_distribution)
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_sign"] = np.sign(np.mean(res.bootstrap_distribution))
        df_bs.to_parquet(f"/glade/campaign/cisl/aiml/cat_flood/bootstrapping_daily/results_{mrms_label}_{county}.parquet")

In [ ]:
mrms_metrics = [c for c in list(df.columns) if "_ffw" in c]
mrms_metrics_bool = [c+b for c in mrms_metrics for b in ["_yes", "_no"]]
mrms_metrics_bool_bs = [c+bs for c in mrms_metrics_bool for bs in ["_pvalue", "_boot_mean", "_boot_sign"]]
deltas = [c for c in list(df.columns) if any(k in c for k in ['_z', '_modz'])]
counties = ["LA", "SB", "VE"]
bools = ["yes", "no"]

for county in counties:
    print(county)
    df_bs = pd.DataFrame(columns=mrms_metrics_bool_bs, index=deltas)
    for mrms_metric in mrms_metrics:
        print(mrms_metric)
        for delta in deltas:
            print(delta)
            for bool in bools:
                data = df.loc[(df[mrms_metric]==bool) & (df['county']==county)][delta].dropna()
                if data.shape[0]<2:
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = np.nan
                    continue
                else:
                    res = bootstrap((data,), np.mean, confidence_level=0.95, paired=False, alternative='two-sided')
                    res_ecdf = ecdf(res.bootstrap_distribution)
                    res_ecdf_0 = float(res_ecdf.cdf.evaluate(0))
                    if res_ecdf_0 > 0.5:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = 1.0 - res_ecdf_0
                    else:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = res_ecdf_0
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_mean"] = np.mean(res.bootstrap_distribution)
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_sign"] = np.sign(np.mean(res.bootstrap_distribution))
    df_bs.to_parquet(f"bootstrapping_daily/results_ffw_{county}.parquet")